In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

%matplotlib inline
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

### This code loads the Adult dataset and assigns column names.

In [ ]:
columns = [
    "age","workclass","fnlwgt","education","education-num",
    "marital-status","occupation","relationship","race","sex",
    "capital-gain","capital-loss","hours-per-week",
    "native-country","income"
]

df = pd.read_csv("adult.data", names=columns, sep=",", skipinitialspace=True)
df.head()

### This code displays the first 5 rows to confirm the dataset was loaded correctly.

In [ ]:
df.head()

### This code shows the dataset structure, column types, and non-null counts.

In [ ]:
df.info()

### This code displays descriptive statistics for the numerical columns.

In [ ]:
df.describe()

### This code replaces '?' values with missing values (NaN).

In [ ]:
df.replace("?", np.nan, inplace=True)
df.head()

### This code checks the number of missing values in each column.

In [ ]:
df.isnull().sum()

### This code visualizes missing values using a heatmap.

In [ ]:
plt.figure(figsize=(10,5))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.show()

### This code plots the target variable distribution for income.

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='income', data=df)
plt.show()

### This code plots the income distribution grouped by sex.

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='income', hue='sex', data=df)
plt.show()

### This code plots the age distribution using a histogram.

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['age'].dropna(), bins=30)
plt.show()

### This code creates a scatter joint plot between age and hours-per-week.

In [ ]:
sns.jointplot(x='age', y='hours-per-week', data=df, kind='scatter')
plt.show()

### This code creates a KDE joint plot between education-num and age.

In [ ]:
sns.jointplot(x='education-num', y='age', data=df, kind='kde', fill=True)
plt.show()

### This code creates a scatter joint plot between hours-per-week and capital-gain.

In [ ]:
sns.jointplot(x='hours-per-week', y='capital-gain', data=df, kind='scatter')
plt.show()

### This code creates a pairplot for selected features colored by income.

In [ ]:
pairplot_df = df[['age','education-num','hours-per-week','capital-gain','income']].dropna()
sns.pairplot(pairplot_df, hue='income')
plt.show()

### This code separates the predictors (X) from the target variable (y).

In [ ]:
X = df.drop('income', axis=1)
y = df['income']

X.head()

### This code identifies categorical and numerical columns for preprocessing.

In [ ]:
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

### This code splits the dataset into training and testing sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

### This code builds a preprocessing and modeling pipeline using imputation, one-hot encoding, and logistic regression.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), numeric_cols)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

### This code uses the trained model to generate predictions on the test set.

In [ ]:
predictions = model.predict(X_test)
predictions[:10]

### This code calculates the model accuracy score.

In [ ]:
accuracy_score(y_test, predictions)

### This code prints the classification report including precision, recall, and F1-score.

In [ ]:
print(classification_report(y_test, predictions))

### This code computes the confusion matrix.

In [ ]:
cm = confusion_matrix(y_test, predictions, labels=model.named_steps['classifier'].classes_)
cm

### This code visualizes the confusion matrix using a heatmap.

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=model.named_steps['classifier'].classes_,
    yticklabels=model.named_steps['classifier'].classes_
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()